# CRA-GQA — Kaggle Inference Smoke Test

Runs CRA **inference/evaluation** on a tiny synthetic NextGQA subset.

This notebook is self-contained:
1. Builds synthetic data
2. Trains for **1 epoch** to produce a checkpoint (fast)
3. Reloads the checkpoint and calls `pipeline.infer()`

**Kaggle settings**
- Accelerator: **GPU**
- Internet: **On**

If you already ran `kaggle_training_smoke_test.ipynb`, you can skip the training cell and point `RESUME_CKPT` at the saved checkpoint.

In [ ]:
import os
from pathlib import Path

import torch

os.environ.setdefault("HF_HOME", "/kaggle/working/hf_cache")
os.environ.setdefault("TRANSFORMERS_CACHE", "/kaggle/working/hf_cache")

if not torch.cuda.is_available():
    raise RuntimeError("Enable the GPU accelerator in Kaggle notebook settings.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from kaggle_helpers.kaggle_utils import ensure_repo, install_smoke_deps, patch_roberta_paths

REPO_ROOT = ensure_repo()
install_smoke_deps()
patch_roberta_paths()
print("Repo root:", REPO_ROOT)

In [ ]:
from kaggle_helpers.smoke_data import setup_smoke_dataset
from kaggle_helpers.kaggle_utils import load_smoke_config
from pipeline import pipeline_fns
from utils.misc import setup_seed

paths = setup_smoke_dataset(root=REPO_ROOT / "data", max_feats=8)

RUN_NAME = "kaggle/infer_smoke"
OUTPUT_DIR = "/kaggle/working/output"
CKPT_DIR = Path(OUTPUT_DIR) / RUN_NAME / "checkpoint"
RESUME_CKPT = CKPT_DIR / "model_best.pth"

cfgs = load_smoke_config(
    cfg_path=str(REPO_ROOT / "config/CRA/CRA_NextGQA.yml"),
    running_name=RUN_NAME,
    max_feats=8,
    batch_size=2,
    epochs=1,
    record_dir=OUTPUT_DIR,
    resume=None,
)

setup_seed(cfgs["misc"]["seed"])

In [ ]:
# Train briefly if no checkpoint exists yet (keeps this notebook self-contained on Kaggle)
if not RESUME_CKPT.exists():
    alt_ckpt = Path(OUTPUT_DIR) / "kaggle/train_smoke/checkpoint/model_best.pth"
    if alt_ckpt.exists():
        RESUME_CKPT = alt_ckpt
        print("Reusing checkpoint from training smoke test:", RESUME_CKPT)
    else:
        print("No checkpoint found — running 1-epoch mini-train...")
        train_pipeline = pipeline_fns[cfgs["optim"]["pipeline"]](cfgs)
        train_pipeline.train()
        if not RESUME_CKPT.exists():
            RESUME_CKPT = CKPT_DIR / "current_checkpoint.pth"
        print("Checkpoint ready:", RESUME_CKPT)
else:
    print("Checkpoint already exists:", RESUME_CKPT)

assert RESUME_CKPT.exists(), f"Missing checkpoint: {RESUME_CKPT}"

In [ ]:
cfgs_infer = load_smoke_config(
    cfg_path=str(REPO_ROOT / "config/CRA/CRA_NextGQA.yml"),
    running_name=RUN_NAME + "_eval",
    max_feats=8,
    batch_size=2,
    epochs=1,
    record_dir=OUTPUT_DIR,
    resume=str(RESUME_CKPT),
)

setup_seed(cfgs_infer["misc"]["seed"])
infer_pipeline = pipeline_fns[cfgs_infer["optim"]["pipeline"]](cfgs_infer)

scores = infer_pipeline.infer()
print("\nInference smoke test finished.")
print("Returned score keys:", sorted(scores.keys()))

required = ["val_acc", "test_acc"]
for key in required:
    assert key in scores, f"Missing metric: {key}"
    print(f"{key}: {scores[key]:.4f}")

print("Inference smoke test passed.")

## Notes

- Metrics are computed on synthetic data, so accuracy/IoU numbers are not meaningful — the goal is to verify the inference path runs end-to-end.
- For **TempCLIP**, call `pipeline.inference()` instead of `pipeline.infer()`.
- For real NextGQA evaluation, replace the synthetic data with the full dataset and a trained checkpoint.